# Phase 4 v0.5 Round — Kaggle CUDA

v0.5 변경사항: Mode 3 / Image 모달리티 삭제, Mode1Head in_dim 384→256.
데이터는 v0.4와 동일 (`phase4_v0_4.h5` 20-dim). Kaggle Dataset: `donghyun51/lens-phase4-v0-4`.

실행 전 Datasets 탭에서 `donghyun51/lens-phase4-v0-4` 추가 확인.


In [ ]:
import os, subprocess

ROUND_SCRIPT = "scripts/phase4_v0_5_round.py"
# 데이터 파일명은 v0.4 그대로 (재업로드 불필요)
DATA_NAME    = "phase4_v0_4.h5"
UNFILTERED_NAME = "phase4_v0_4_eval_unfiltered.h5"
SCALER_NAME  = "target_scaler_phase4_v0_4.pkl"

REPO_URL = "https://github.com/dasbaq/GV.git"
REPO_DIR = "/kaggle/working/repo"
print(subprocess.check_output(["nvidia-smi"], text=True))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


In [ ]:
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/repo/Gravitational_Lens_MultiMode


In [ ]:
from pathlib import Path
h5s  = sorted(Path('/kaggle/input').rglob('*.h5'))
pkls = sorted(Path('/kaggle/input').rglob('*.pkl'))
print('H5 files:');  [print(' ', p) for p in h5s]
print('PKL files:'); [print(' ', p) for p in pkls]

train      = next(p for p in h5s  if p.name == DATA_NAME)
unfiltered = next((p for p in h5s  if p.name == UNFILTERED_NAME), None)
scaler     = next((p for p in pkls if p.name == SCALER_NAME), None)
# equivalence JSON이 있으면 --equivalence-from 인자로 넘긴다
equivalence = next(
    (p for p in sorted(Path('/kaggle/input').rglob('*equivalence.json'))
     if 'phase4_v0_5' in p.name or 'phase4_v0_4' in p.name),
    None,
)

os.environ['LENS_DATA_PATH']             = str(train)
os.environ['LENS_DATA_PATH_UNFILTERED']  = str(unfiltered) if unfiltered else ''
os.environ['LENS_SCALER_PATH']           = str(scaler)     if scaler     else ''
os.environ['LENS_WORK_ROOT']             = '/kaggle/working'

print('LENS_DATA_PATH=',            os.environ['LENS_DATA_PATH'])
print('LENS_DATA_PATH_UNFILTERED=', os.environ.get('LENS_DATA_PATH_UNFILTERED'))
print('LENS_SCALER_PATH=',          os.environ.get('LENS_SCALER_PATH'))
print('equivalence=',               equivalence)


In [ ]:
# 2-epoch 빠른 sanity — acceptance/leak gate는 10 epoch 미만이라 skip됨
eq_arg = f"--equivalence-from {equivalence}" if equivalence else ""
!python {ROUND_SCRIPT} --phase train {eq_arg} --device cuda --workers 0 --epochs 2 --bootstrap-n 0


In [ ]:
# Full run (workers=0: v0.4에서 workers=4보다 빠름 확인됨)
!python {ROUND_SCRIPT} --phase train {eq_arg} --device cuda --workers 0 --epochs 50 --bootstrap-n 1000


In [ ]:
import json
for p in sorted(Path('/kaggle/working/logs').glob('*.json')):
    print('\n==', p.name, '==')
    data = json.loads(p.read_text())
    if 'stage_b_acceptance_report' in data:
        print(json.dumps(data['stage_b_acceptance_report'], indent=2)[:4000])
    elif 'best' in data:
        print(json.dumps(data.get('best', {}), indent=2)[:4000])
    else:
        print(json.dumps(data, indent=2)[:2000])


In [ ]:
# checkpoint head1.net.0.weight shape 확인 (v0.5 = [64, 256], v0.4 = [64, 384])
import torch
ckpt_path = next(Path('/kaggle/working/checkpoints').glob('phase4_v0_5_imgres_best.pt'), None)
if ckpt_path:
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    w = sd.get('head1.net.0.weight')
    print(f'head1.net.0.weight shape: {w.shape if w is not None else None}')
    print(f'par_enc.net.0.weight shape: {sd.get("par_enc.net.0.weight", {"shape": "missing"}).shape}')
    # img_enc 키가 없어야 함 (v0.5)
    img_keys = [k for k in sd if k.startswith('img_enc.')]
    print(f'img_enc keys (should be 0): {len(img_keys)}')
    assert w is not None and w.shape == (64, 256), f'expected (64,256), got {w.shape}'
    assert len(img_keys) == 0, f'img_enc keys found: {img_keys[:3]}'
    print('v0.5 checkpoint verified OK')
else:
    print('checkpoint not found')
